# 01. Zero Trust Fundamentals

## Background

The perimeter security model dominated enterprise security for decades: firewall the edge, trust everything inside. This model broke down as cloud infrastructure dissolved the perimeter, SaaS moved corporate data to third-party networks, and lateral movement became the dominant attack pattern. The 2020 SolarWinds breach demonstrated why implicit internal trust is fatal.

**Zero Trust** is the architectural response: never trust, always verify, assume breach. Every request is authenticated and authorized regardless of network origin. Microsegmentation limits blast radius. All access is logged, minimal, and continuously re-evaluated.

## What You'll Learn

- Zero trust principles: verify explicitly, use least privilege, assume breach
- PDP (Policy Decision Point) and PEP (Policy Enforcement Point) architecture
- Trust scoring: combining device, identity, and behavioral signals into a continuous score
- Building a zero trust policy evaluator in Python
- Microsegmentation and blast radius analysis

## Why This Matters

Zero trust is the baseline expectation for enterprise and cloud-native security (NIST SP 800-207, Google BeyondCorp, Cloudflare Zero Trust). LLM applications face the same challenge: every tool call is an access decision that must be authorized based on context, not just session presence. The `ZeroTrustPolicyEngine` built here is the pattern behind every modern zero trust PDP.

In [ ]:
import hashlib, hmac, time, json, re
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Trust Scoring Engine

Zero trust replaces binary in/out with a continuous trust score built from multiple signals: identity (MFA), device posture (MDM enrollment, health attestation), network context, and behavioral analytics (anomaly score, geo-risk).

In [ ]:
class TrustLevel(Enum):
    DENY   = 0
    LOW    = 1
    MEDIUM = 2
    HIGH   = 3
    FULL   = 4


@dataclass
class RequestContext:
    user_id:         str
    resource:        str
    action:          str
    device_managed:  bool  = False
    device_healthy:  bool  = False
    mfa_verified:    bool  = False
    network_trusted: bool  = False
    anomaly_score:   float = 0.0
    hour_of_day:     int   = 12
    geo_risk:        float = 0.0


@dataclass
class PolicyResult:
    allowed:     bool
    trust_level: TrustLevel
    trust_score: float
    reasons:     list = field(default_factory=list)
    conditions:  list = field(default_factory=list)


class ZeroTrustPolicyEngine:
    RESOURCE_POLICY = {
        'public_api':     TrustLevel.LOW,
        'internal_api':   TrustLevel.MEDIUM,
        'sensitive_data': TrustLevel.HIGH,
        'admin_console':  TrustLevel.FULL,
        'llm_tool_call':  TrustLevel.MEDIUM,
    }

    def _score(self, ctx):
        s, r = 0.0, []
        if ctx.mfa_verified:      s += 0.30; r.append('+0.30 MFA')
        if ctx.device_managed:    s += 0.20; r.append('+0.20 managed device')
        if ctx.device_healthy:    s += 0.15; r.append('+0.15 device healthy')
        if ctx.network_trusted:   s += 0.10; r.append('+0.10 trusted network')
        pen = ctx.anomaly_score * 0.40; s -= pen
        if pen > 0:               r.append(f'-{pen:.2f} anomaly')
        geo = ctx.geo_risk * 0.20; s -= geo
        if geo > 0.05:            r.append(f'-{geo:.2f} geo risk')
        if 8 <= ctx.hour_of_day <= 18: s += 0.05; r.append('+0.05 business hours')
        return max(0.0, min(1.0, s)), r

    def _level(self, s):
        if s >= 0.80: return TrustLevel.FULL
        if s >= 0.60: return TrustLevel.HIGH
        if s >= 0.40: return TrustLevel.MEDIUM
        if s >= 0.20: return TrustLevel.LOW
        return TrustLevel.DENY

    def evaluate(self, ctx):
        score, reasons = self._score(ctx)
        level = self._level(score)
        required = self.RESOURCE_POLICY.get(ctx.resource, TrustLevel.HIGH)
        allowed = level.value >= required.value
        conditions = []
        if not allowed:
            if not ctx.mfa_verified:     conditions.append('Complete MFA step-up')
            if not ctx.device_managed:   conditions.append('Enroll device in MDM')
            if ctx.anomaly_score > 0.3:  conditions.append('Anomalous activity flagged')
        return PolicyResult(allowed, level, score, reasons, conditions)


engine = ZeroTrustPolicyEngine()
scenarios = [
    ('Admin — full trust',
     RequestContext('alice','admin_console','write',True,True,True,True,0.0,10,0.0)),
    ('Sensitive data — no MFA',
     RequestContext('bob','sensitive_data','read',True,True,False,False,0.0,14,0.1)),
    ('Internal API — anomalous behavior',
     RequestContext('charlie','internal_api','read',False,False,True,False,0.8,3,0.5)),
    ('LLM tool call — normal session',
     RequestContext('dana','llm_tool_call','execute',True,True,True,False,0.1,15,0.0)),
]
for label, ctx in scenarios:
    r = engine.evaluate(ctx)
    status = 'ALLOW' if r.allowed else 'DENY '
    print(f"[{status}] {label}")
    print(f"        score={r.trust_score:.2f}  level={r.trust_level.name}")
    if r.conditions:
        for c in r.conditions: print(f"        step-up: {c}")
    print()

In [ ]:
# Trust score sensitivity: MFA vs anomaly
anomaly_vals = np.linspace(0, 1, 25)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, managed in zip(axes, [False, True]):
    grid = np.zeros((2, len(anomaly_vals)))
    for i, mfa in enumerate([False, True]):
        for j, anom in enumerate(anomaly_vals):
            ctx = RequestContext('x','internal_api','read',
                                 device_managed=managed, device_healthy=managed,
                                 mfa_verified=mfa, anomaly_score=anom, hour_of_day=12)
            grid[i, j] = engine.evaluate(ctx).trust_score
    im = ax.imshow(grid, aspect='auto', vmin=0, vmax=1, cmap='RdYlGn',
                   extent=[0,1,-0.5,1.5])
    ax.set_yticks([0,1]); ax.set_yticklabels(['MFA=No','MFA=Yes'])
    ax.set_xlabel('Anomaly Score')
    ax.set_title(f'Trust Score — device_managed={managed}')
    plt.colorbar(im, ax=ax, label='Trust Score')
plt.suptitle('Zero Trust Score Sensitivity (internal_api)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('01_trust_heatmap.png', dpi=110, bbox_inches='tight')
plt.show()

## 2. Microsegmentation and Blast Radius

Zero trust limits lateral movement by explicitly defining which roles can reach which resources. A compromised `analyst` account cannot pivot to `admin_console` even from inside the network — there is no implicit path.

In [ ]:
resources = ['public_api','internal_api','sensitive_data','admin_console','llm_tool_call']
users = ['analyst','developer','admin','external_svc']
zt = {
    'analyst':     [True, True, False,False,True],
    'developer':   [True, True, False,False,True],
    'admin':       [True, True, True, True, True],
    'external_svc':[True, False,False,False,False],
}
flat = {u: [True]*5 for u in users}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, policy, title in [(axes[0],flat,'Traditional: Flat Network'),
                           (axes[1],zt,  'Zero Trust: Microsegmented')]:
    mat = [policy[u] for u in users]
    ax.imshow(mat, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(5)); ax.set_xticklabels(resources, rotation=30, ha='right')
    ax.set_yticks(range(4)); ax.set_yticklabels(users)
    ax.set_title(title)
    for i in range(4):
        for j in range(5):
            ax.text(j,i,'Y' if mat[i][j] else 'N',ha='center',va='center',
                    color='white',fontweight='bold',fontsize=11)
plt.suptitle('Blast Radius: Traditional vs Zero Trust', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('01_microsegmentation.png', dpi=110, bbox_inches='tight')
plt.show()
flat_n = sum(sum(v) for v in flat.values())
zt_n   = sum(sum(v) for v in zt.values())
print(f"Flat network: {flat_n} access paths  |  Zero trust: {zt_n} paths  "
      f"({100*zt_n/flat_n:.0f}% of flat, {flat_n-zt_n} lateral paths closed)")